## Data
Here the artificial data is made like in the paper (well excluding translations for now), T's and L's. Each shape is rotated 4 times and `n_copies` is how many noisy versions of each rotation are generated.

Pixel values are scaled to $[−\pi, \pi]$ for angle encoding.

In [ ]:
N_LAYERS = 5
N_STEPS = 20
LR = 0.1
N_COPIES = 10
NOISE_STD = 0.5
TRAIN_FRAC = 0.75
SEED = 0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

T = np.array([
    [1, 1, 1, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 0],
], dtype=float)

L = np.array([
    [0, 1, 0, 0],
    [0, 1, 0, 0],
    [0, 1, 1, 0],
    [0, 0, 0, 0],
], dtype=float)

def make_dataset(n=4, n_copies=10, noise_std=0.1, seed=0):
    """T -> +1, L -> -1. N = 2 * 4_rotations * n_copies."""
    rng = np.random.default_rng(seed)
    X, y = [], []
    for template, label in [(T, 1.0), (L, -1.0)]:
        for k in range(4):
            base = np.rot90(template, k) * np.pi # scale to [-π, π] for RX encoding
            for _ in range(n_copies):
                noise = rng.normal(0, noise_std, (n, n))
                X.append(np.clip(base + noise, -np.pi, np.pi).flatten())
                y.append(label)
    return np.array(X, dtype=float), np.array(y, dtype=float)

def view_samples(X, y, n=5, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=n, replace=False)
    fig, axes = plt.subplots(1, n, figsize=(2 * n, 2))
    for ax, i in zip(axes, idx):
        ax.imshow(X[i].reshape(4, 4), cmap="grey", vmin=-np.pi, vmax=np.pi)
        ax.set_title("T" if y[i] == 1 else "L")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
X, y = make_dataset(n_copies=N_COPIES, noise_std=NOISE_STD, seed=SEED)

rng = np.random.default_rng(SEED)
idx = rng.permutation(len(X))
split = int(len(X) * TRAIN_FRAC)
X_train, y_train = X[idx[:split]], y[idx[:split]]
X_test,  y_test  = X[idx[split:]], y[idx[split:]]

print(f"Train: {len(X_train)}  Test: {len(X_test)}")
print(f"Label distribution — train: T={int((y_train==1).sum())} L={int((y_train==-1).sum())}")
print(f"Label distribution — test:  T={int((y_test==1).sum())}  L={int((y_test==-1).sum())}")

view_samples(X, y)

### C₄ orbits
Equation 9 from the paper `(i,j)` → `(n−j−1, i)` shows what each rotation does to the pixel. So the $4\times 4$ picture has the orbits:
```
orbit 0: [ 0, 12, 15,  3]
orbit 1: [ 1,  8, 14,  7]
orbit 2: [ 2,  4, 13, 11]
orbit 3: [ 5,  9, 10,  6]
```
### Twirling
The Twirling formula $T[A] = \frac{1}{|G|} \sum_g U_g A U_g^\dagger$ symmetrizes any gate to commute with all group elements. The orbits will share parameters and so `qml.Rot` is applied by orbit with $(\alpha, \beta, \gamma)$ as the parameters
### CNOT entanglement
1. **Intra-cell ring:** at the position `p` in an orbit, ring-connect one qubit from each orbit
2. **Inter-cell ring:** connect `orbit-0[p]` $\rightarrow$ `orbit-1[(p+1) % 4]` $\rightarrow$ stitches rings diagonally
### Measurement
Average Pauli-Z measurement over all the elements in an orbit. These elements map to each other under group action (making it label invariant)


In [ ]:
import pennylane as qml
import pennylane.numpy as pnp

n = 4
n_q = n*n

def compute_orbits(n):
    # C4 rotation means orbit = 4 pixels related by 90 degree steps
    # Twirling with the C4 SWAP unitary forces same params across each orbit
    seen = set()
    orbits = []
    for i in range(n):
        for j in range(n):
            if (i, j) not in seen:
                orbit = [
                    (i, j),
                    (n - j - 1, i),
                    (n - i - 1, n - j - 1),
                    (j, n - i - 1),
                ]
                orbits.append([i_ * n + j_ for i_, j_ in orbit])
                seen.update(orbit)
    return orbits

def encode(x, n_q):
    # Angle encoding: pixel x[k] in [-π, π] -> RX(x[k]) on qubit k.
    for k in range(n_q):
        qml.RX(x[k], wires=k)

def variational_block(params, orbits):
    """params shape: (n_phi, 3), n_phi is the number of orbits, 3 represents the euler angles"""
    for orbit_idx, orbit in enumerate(orbits):
        # Every element in an orbit shares parameters
        alpha, beta, gamma = params[orbit_idx]
        for qubit in orbit:
            # Applies the rotation with shared params to every element in the orbit
            qml.Rot(alpha, beta, gamma, wires=qubit)

    n_phi = len(orbits)
    # Intra-Cell: entangles qubits at the same rotational position across all orbits.
    for p in range(len(orbits[0])):
        for k in range(n_phi):
            qml.CNOT(wires=[orbits[k][p], orbits[(k + 1) % n_phi][p]])
    # Inter-cell: entangles the cells
    for p in range(len(orbits[0])):
        qml.CNOT(wires=[orbits[0][p], orbits[1][(p + 1) % len(orbits[0])]])

orbits = compute_orbits(n)
corner_obs = 0.25 * (qml.PauliZ(0) + qml.PauliZ(3) + qml.PauliZ(12) + qml.PauliZ(15))

print("Orbits: ", orbits)

## Training

In [ ]:
import time

def make_vqc():
    # default.qubit, lightning.qubit
    dev = qml.device("lightning.qubit", wires=n_q)

    @qml.qnode(dev, diff_method="adjoint")
    def vqc(x, params):
        for layer_params in params:
            encode(x, n_q)
            variational_block(layer_params, orbits)
        return qml.expval(corner_obs) # Measure the corner orbit, but any orbit will do

    return vqc

def train(X_tr, y_tr, n_layers=5, n_steps=100, lr=0.1, seed=0):
    rng= np.random.default_rng(seed)
    n_phi = len(orbits)

    # params are the three Euler angles, so 3 params per orbit per layer
    # For the 4x4 image with 5 layers: 3*4*5 = 60 params
    params = pnp.array(
        rng.uniform(0, 2 * np.pi, (n_layers, n_phi, 3)),
        requires_grad=True,
    ) # Random initialization of the 3 parameters per gate per layer
    y_arr = pnp.array(y_tr, requires_grad=False)
    vqc = make_vqc()
    opt = qml.AdamOptimizer(stepsize=lr)
    losses = []

    print(f"Training {n_layers} layers, {n_steps} steps, lr={lr}")
    print(f"Dataset: {len(X_tr)} train samples")
    t_start = time.time()

    for step in range(n_steps):
        def cost(p, X=X_tr, y=y_arr):
            # This is costly since it calls vqc on every item in the training set
            # Fine for this small dataset, but if we scale, needs changed
            preds = pnp.stack([vqc(x, p) for x in X])
            return pnp.mean((preds - y) ** 2)

        params, loss_val = opt.step_and_cost(cost, params)
        losses.append(float(loss_val))

        if step % 10 == 0 or step == n_steps - 1:
            elapsed = time.time() - t_start
            eta = elapsed / (step + 1) * (n_steps - step - 1)
            print(f"  step {step:3d} loss={float(loss_val):.4f}")
            print(f"  elapsed={elapsed/60:.1f}min  eta={eta/60:.1f}min")

    print(f"Done. Total: {(time.time()-t_start)/60:.1f} min")
    return params, losses

## Model Visualization

In [ ]:
x_draw = np.zeros(n_q)
p_draw = np.zeros((1, len(orbits), 3))
vqc_draw = make_vqc()
fig, ax = qml.draw_mpl(vqc_draw)(x_draw, p_draw)
fig.show()

## Evaluation

In [ ]:
def predict(X, params, vqc):
    return np.array([float(vqc(x, params)) for x in X])

def metrics(y_true, y_raw):
    # Same metrics used in the original paper
    pred = np.sign(y_raw)
    acc = float(np.mean(pred == y_true))
    tp = int(np.sum((pred == 1) & (y_true == 1)))
    fp = int(np.sum((pred == 1) & (y_true == -1)))
    fn = int(np.sum((pred == -1) & (y_true == 1)))
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return {"accuracy": acc, "f1": f1, "precision": prec, "recall": rec}

## Train

In [ ]:
params, losses = train(
    X_train, y_train,
    n_layers=N_LAYERS,
    n_steps=N_STEPS,
    lr=LR,
    seed=SEED,
)

## Eval

In [ ]:
vqc_eval = make_vqc()

train_raw  = predict(X_train, params, vqc_eval)
test_raw   = predict(X_test,  params, vqc_eval)
train_m    = metrics(y_train, train_raw)
test_m     = metrics(y_test,  test_raw)

print("\nResults")
print("-" * 50)
for split_name, m in [("Train", train_m), ("Test", test_m)]:
    print(f"{split_name:5s}  acc={m['accuracy']:.3f}  f1={m['f1']:.3f}"
          f"  prec={m['precision']:.3f}  rec={m['recall']:.3f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(losses)
plt.xlabel("Step")
plt.ylabel("MSE loss")
plt.title(f"{N_LAYERS} layers, lr={LR}")
plt.tight_layout()
plt.show()